# RIASEC ANALYSIS

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [2]:
#df = pd.read_csv("RIASEC.csv")

## Reading and Cleaning

In [3]:
df = pd.read_csv("RIASEC.csv", sep=None, engine="python")

print(df.shape)
print(df.head())

(8855, 55)
   implementation  R1  R2  R3  R4  R5  R6  R7  R8  I1  ...  C5  C6  C7  C8  \
0               2   3   1   4   2   1   2   1   1   5  ...   2   1   1   2   
1               2   1   1   1   1   1   1   1   1   4  ...   1   1   1   1   
2               2   3   2   1   1   1   1   2   1   5  ...   3   4   4   4   
3               2   3   2   1   2   2   3   1   2   5  ...   1   3   2   1   
4               2  -1   2   3   2   3   2   1   3   5  ...   4   3   3   3   

   accuracy  elapse  country  fromsearch  age  gender  
0        90     222       PT           0   -1      -1  
1       100     102       US           0   -1      -1  
2        95     264       US           1   -1      -1  
3        60     189       SG           0   -1      -1  
4        90     197       US           0   -1      -1  

[5 rows x 55 columns]


In [4]:
cols = [f"R{i}" for i in range(1, 9)]
print(cols)
R_data = df[cols]
R_data.shape

['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8']


(8855, 8)

In [5]:
df = R_data[(R_data != -1).all(axis=1)] # cleans the data and gets rid of rows with -1 entries
print("Shape after cleaning:", df.shape)

Shape after cleaning: (8478, 8)


In [6]:
# the entries have reduced from 8855 people to 8478 meaning 377 people had invalid answers

In [7]:
8855-8478

377

In [8]:
df

,R1,R2,R3,R4,R5,R6,R7,R8
0,3,1,4,2,1,2,1,1
1,1,1,1,1,1,1,1,1
2,3,2,1,1,1,1,2,1
3,3,2,1,2,2,3,1,2
5,3,1,3,4,3,4,3,3
...,...,...,...,...,...,...,...,...
8849,3,3,1,4,2,2,2,1
8851,3,2,3,4,3,2,2,2
8852,4,3,3,3,2,2,1,2
8853,4,4,3,5,4,5,3,4


In [9]:
df["R1"]

0       3
1       1
2       3
3       3
5       3
       ..
8849    3
8851    3
8852    4
8853    4
8854    4
Name: R1, Length: 8478, dtype: int64

## Model selection

In [10]:
df = df.copy()
df["R_score"] = df.mean(axis=1) #row wise mean
df.shape

(8478, 9)

In [11]:
train = df.iloc[:6500]
test = df.iloc[6500:]
train.shape, test.shape

((6500, 9), (1978, 9))

In [12]:
x_train = train[["R1"]]
y_train = train["R_score"]

In [13]:
model = LinearRegression()
model.fit(x_train, y_train)

LinearRegression()

In [14]:
print("Intercept:", model.intercept_)
print("Coefficient for R1:", model.coef_[0])

model.coef_.shape

Intercept: 1.018105231431332
Coefficient for R1: 0.42359934763433976


(1,)

Hence the form of the best-fit regression line is $R_{score} = R_1 \times \text{coef} + \text{intercept}$

$RMSE = \sqrt(\dfrac{\sum_i (R_{score}[i] - \hat{R}_{score}[i])^2}{\sum_i i})$

In [15]:
y_pred_train = model.predict(x_train)
RSS_train = np.sum((y_train - y_pred_train) ** 2) #RSS -> residual sum of squares for variance explanation
MSE_train = RSS_train / len(train)
print("Training RSS (total):", RSS_train)
print("Train MSE:", MSE_train)
print("Train RMSE:", np.sqrt(MSE_train))

Training RSS (total): 2902.0393474685015
Train MSE: 0.446467591918231
Train RMSE: 0.6681823044036942


## Validation

In [16]:
x_test = test[["R1"]]
y_test = test["R_score"]

In [17]:
y_pred_test = model.predict(x_test)
RSS_test = np.sum((y_test - y_pred_test) ** 2)
MSE_test = RSS_test / len(test)

In [18]:
print("Test RSS (total):", RSS_test)
print("Test MSE:", MSE_test)
print("Test RMSE:", np.sqrt(MSE_test))

Test RSS (total): 1028.7757850021012
Test MSE: 0.5201090925187569
Test RMSE: 0.7211858931778664


In [19]:
print(f"RMSE (train): {np.sqrt(MSE_train):.5f}")
print(f"RMSE (test): {np.sqrt(MSE_test):.5f}")

RMSE (train): 0.66818
RMSE (test): 0.72119


Model works better with training data

# For other fields

### Training

In [20]:
models = {}
RSS_train = {}
MSE_train = {}

for i in range(2, 9):   # R2 to R8
    x_train = train[[f"R{i}"]]
    y_train = train["R_score"]

    models[i] = LinearRegression()
    models[i].fit(x_train, y_train)

    y_pred_train = models[i].predict(x_train)

    RSS_train[i] = np.sum((y_train - y_pred_train) ** 2)
    MSE_train[i] = RSS_train[i] / len(train)

    print(f"R{i}:")
    print(f"  Training RSS (total): {RSS_train[i]}")
    print(f"  Training MSE : {MSE_train[i]}")
    print(f"  RMSE : {np.sqrt(MSE_train[i])}\n")

R2:
  Training RSS (total): 2086.9934694791536
  Training MSE : 0.32107591838140825
  RMSE : 0.566635613407248

R3:
  Training RSS (total): 2856.1295910903655
  Training MSE : 0.43940455247544086
  RMSE : 0.66287597065774

R4:
  Training RSS (total): 2058.2069482183156
  Training MSE : 0.31664722280281776
  RMSE : 0.5627141572795354

R5:
  Training RSS (total): 2026.8220877210033
  Training MSE : 0.3118187827263082
  RMSE : 0.5584073627078248

R6:
  Training RSS (total): 1762.7887420860695
  Training MSE : 0.2711982680132415
  RMSE : 0.5207669997352381

R7:
  Training RSS (total): 1928.2557723088821
  Training MSE : 0.29665473420136645
  RMSE : 0.5446602006768683

R8:
  Training RSS (total): 1771.9059588635785
  Training MSE : 0.2726009167482428
  RMSE : 0.5221119772120181



In [21]:
dict(sorted(MSE_train.items(), key=lambda item: item[1]))

{6: np.float64(0.2711982680132415),
 8: np.float64(0.2726009167482428),
 7: np.float64(0.29665473420136645),
 5: np.float64(0.3118187827263082),
 4: np.float64(0.31664722280281776),
 2: np.float64(0.32107591838140825),
 3: np.float64(0.43940455247544086)}

In [22]:
keys = list(MSE_train.keys())
values = list(MSE_train.values())
best_feature = keys[np.argmin(values)]
print(f"Best feature is {best_feature} based on training data")

Best feature is 6 based on training data


### Testing

In [23]:
models = {}
RSS_test = {}
MSE_test = {}

for i in range(2, 9):   # R2 to R8
    x_test = test[[f"R{i}"]]
    y_test = test["R_score"]

    models[i] = LinearRegression()
    models[i].fit(x_test, y_test)

    y_pred_test = models[i].predict(x_test)

    RSS_test[i] = np.sum((y_test - y_pred_test) ** 2)
    MSE_test[i] = RSS_test[i] / len(test)

    print(f"R{i}:")
    print(f"  Testing RSS (total): {RSS_test[i]}")
    print(f"  Testing MSE : {MSE_test[i]}")
    print(f"  RMSE : {np.sqrt(MSE_test[i])}\n")

R2:
  Testing RSS (total): 712.0378480768431
  Testing MSE : 0.35997868962428875
  RMSE : 0.5999822410907583

R3:
  Testing RSS (total): 875.312355121421
  Testing MSE : 0.4425239409107285
  RMSE : 0.6652247296295655

R4:
  Testing RSS (total): 799.8393275744875
  Testing MSE : 0.4043677085816418
  RMSE : 0.6358991339683061

R5:
  Testing RSS (total): 697.1299733893734
  Testing MSE : 0.3524418470118167
  RMSE : 0.5936681286811822

R6:
  Testing RSS (total): 630.9467824594283
  Testing MSE : 0.31898219537888184
  RMSE : 0.5647850877801943

R7:
  Testing RSS (total): 680.1640236301748
  Testing MSE : 0.3438645215521612
  RMSE : 0.586399626152815

R8:
  Testing RSS (total): 678.3905837840575
  Testing MSE : 0.3429679392234871
  RMSE : 0.5856346465361207



In [24]:
dict(sorted(MSE_test.items(), key=lambda item: item[1]))

{6: np.float64(0.31898219537888184),
 8: np.float64(0.3429679392234871),
 7: np.float64(0.3438645215521612),
 5: np.float64(0.3524418470118167),
 2: np.float64(0.35997868962428875),
 4: np.float64(0.4043677085816418),
 3: np.float64(0.4425239409107285)}

In [25]:
keys = list(MSE_test.keys())
values = list(MSE_test.values())
best_feature = keys[np.argmin(values)]
print(f"Best feature is {best_feature} based on testing data")

Best feature is 6 based on testing data


#### We can collectively agree that for mean value based regression, 6 is the best performing fitting parameter.